# # 기초통계 실습

[사용 제한 안내]
> 본 실습 코드는 교육 및 학습 목적으로만 제공됩니다.

[사전 준비 사항]
- 분석 데이타 : house_edu.csv

[분석 요건]
- 분석 주제 : 주택가격 예측 모형 생성 위한 기초 통계 분석
- 분석 단위 : 주택
- 종속 변수 : 주택가격 (price)
- 설명 변수 : 주택정보 (주택면적, 거실수, 층수, 건축기간 등)

[주요 실습 내용]
- 용어 학습 : 변수, 범주형변수 vs 연속형변수, 종속변수 vs 독립변수, 파생변수 등
- 기초 통계량 확인 : 대표값, 표준편차, 사분위수, 왜도, 첨도 등
- 데이타 변환 : Missing 처리, Outlier 처리, 데이터 Scaling, 분포 변환, One-Hot Encoding 등
- 상관 분석
- 회귀 분석


# 한글 폰트 설치

In [ ]:
!sudo apt-get install -y fonts-nanum
!rm -rf ~/.cache/matplotlib

**!! 메뉴에서 "런타임" → "세션 다시 시작"**

# 데이터 준비
- 주택 가격 예측 모델 생성용 공개 데이터셋
- 원본 : https://www.kaggle.com/datasets/harlfoxem/housesalesprediction
- 교육 목적으로 원본 데이터를 일부 변형 시킴
- 변수 설명
| 변수명 | 의미 | 비고 |
|--------|------|------|
| id | 주택 거래 고유 식별자 | |
| date | 주택 판매 날짜 | |
| price | 주택 판매 가격 | **종속변수(Target)** |
| bedrooms | 침실 개수 | |
| living_area | 거실(주거) 면적 | |
| lot_area | 토지 면적 | |
| floors | 층수 | |
| waterfront | 워터뷰 여부 (0=없음, 1=있음) | |
| grade | 건물 품질 및 디자인 등급 | |
| above_area | 지상 면적 | |
| basement_area | 지하실 면적 | |
| yr_built | 건축 연도 | |
| zipcode | 우편번호 | |

데이터 불러오기

In [ ]:
import pandas as pd

df = pd.read_csv("house_edu.csv", encoding="cp949")

데이터 조회하기

In [ ]:
df

불필요 변수 제거

In [ ]:
df = df.drop(columns=["id", "zipcode"])

컬럼정보 확인


In [ ]:
df.info()

[참고] info()에서 확인할 것:
  - 총 행/열 수
  - 각 컬럼의 데이터 타입 (int, float, object 등)
  - int : 연속형 (정수형)
  - float : 연속형 (실수형)
  - object : 보통 문자열 (범주형)
  - category : 범주형,  *범주 갯수가 적을 경우 메모리 효율화를 위해 object 타입 대신 사용, 그러나 object 을 사용해도 무방
  - datetime : 날짜형

컬럼타입 변경

In [ ]:
df["date"] = pd.to_datetime(df["date"])                   # 문자열 -> 날짜 type으로 변환
df["waterfront"] = df["waterfront"].astype("category")    # 실수형 -> 범주형으로 변환

컬럼타입 재확인

In [ ]:
df.info()

### [Self-Practice]

- 실습 주제 : 생성형 AI 에게 데이타 정보를 제공하고 파생변수 추천 요청하기
- 실습 조건
  - 위 df.info() 결과에서 컬럼 정보를 Copy 해서 생성형 AI 에게 input 으로 제공  
- 실습 시나리오
   - 시나리오 1) 생성형 AI 에게 파생변수 추천 요청하기
   - 시나리오 2) 생성형 AI가 추천한 변수 의미 파악하기

# 파생변수 생성

주택 나이 생성

In [ ]:
df["house_age"] = df["date"].dt.year - df["yr_built"]   # 주택 판매 날짜 - 건축 연도

방 밀도 생성

In [ ]:
import numpy as np

df["room_density"] = df["living_area"] / df["bedrooms"].replace(0, np.nan)   # 거실 면적 / 침실 개수

Date 컬럼 제거

In [ ]:
df = df.drop(columns=["date"])  #주택 판매 날짜

# EDA

연속형 변수 기본 분포값 확인
 - 비즈니스관점에서 데이터 이상여부 확인, 특히 음수(-), '0' 데이터 주의
 - Missing 건수 확인
 - 데이터 최소, 최대, 중심값 확인
 - 데이터 밀집도 확인

In [ ]:
import numpy as np

# 연속형 변수 분포 확인
df.select_dtypes(include=[np.number]).describe()

연속형 변수 분포 그래프 확인
 - 히스토그램 (Histogram) 차트 : 퍼짐, 쏠림, 정규분포 확인

In [ ]:
import numpy as np

# 연속형 변수 자동 추출
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("연속형 변수:", num_cols)

# 빈도 분포 히스토그램 (Frequency Histogram)
import matplotlib.pyplot as plt

n = len(num_cols)
cols = 3     # 한 행에 그래프 갯수
rows = (n // cols) + (n % cols > 0)

plt.figure(figsize=(cols*5, rows*4))

for i, col in enumerate(num_cols, 1):
    plt.subplot(rows, cols, i)
    plt.hist(df[col].dropna(), bins=20)
    plt.title(f"{col} Frequency")
    plt.xlabel(col)
    plt.ylabel("Count")

plt.tight_layout()
plt.show()

 - 박스 플롯(Box Plot) : 사분위수 범위 (IQR), 이상치 후보 확인

In [ ]:
import numpy as np

# 연속형 변수 자동 추출
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("연속형 변수:", num_cols)

# 박스 플롯(Box Plot)
cols = 3    # 한 행에 그래프 갯수
rows = (len(num_cols) // cols) + (len(num_cols) % cols > 0)

plt.figure(figsize=(cols*5, rows*4))

for i, col in enumerate(num_cols, 1):
    plt.subplot(rows, cols, i)
    plt.boxplot(df[col].dropna())
    plt.title(col)
    plt.ylabel("Value")

plt.tight_layout()
plt.show()


 - 왜도, 첨도

In [ ]:
import numpy as np

# 연속형 변수 자동 추출
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("연속형 변수:", num_cols)

# 왜도 & 첨도 계산
skew_kurt = pd.DataFrame({
    "Skewness": df[num_cols].skew(),     # 왜도 (>0: 왼쪽으로 치우침 (꼬리가 오른쪽), <0: 오른쪽으로 치우팀 (꼬리가 왼쪽), 0=좌우 대칭)
    "Kurtosis": df[num_cols].kurt()      # 첨도 (>0: 뾰족, <0: 납작, 0=정규분포)
})

print(skew_kurt.sort_values(by="Skewness", ascending=False))

범주형 변수 기본 분포값 확인
 - 비즈니스관점에서 데이터 이상여부 확인
 - Missing 건수 확인
 - 데이터 밀집도 확인
 - 범주 갯수 확인

In [ ]:
# 범주형 변수 자동 추출
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 범주 건수 & 비율 계산
for col in cat_cols:
    print(f"\n[{col}]")
    dist = df[col].value_counts(dropna=False)
    ratio = df[col].value_counts(normalize=True, dropna=False) * 100
    result = pd.DataFrame({
        "count": dist,
        "ratio_%": ratio.round(2)
    })
    print(result)

범주형 변수 분포 그래프 확인
 - 히스토그램 (Histogram) 차트

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 범주형 변수 찾기
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 시각화
for col in cat_cols:
    plt.figure(figsize=(10, 6))

    # 막대 그래프
    df[col].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')

    plt.title(f'{col} 빈도 분포', fontsize=14, fontweight='bold')
    plt.xlabel(col, fontsize=12)
    plt.ylabel('빈도', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

목표변수 vs 설명변수 관계 확인
- 연속형 설명변수 산점도(Scatter Plot)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 연속형 변수 추출
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# 목표변수(price) 제외
feature_cols = [col for col in num_cols if col != "price"]

# Scatter Plot
n_cols = 3
n_rows = int(np.ceil(len(feature_cols) / n_cols))
plt.figure(figsize=(15, 4 * n_rows))

for i, col in enumerate(feature_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.scatterplot(data=df, x=col, y="price", alpha=0.5)
    plt.title(f"price vs {col}")

plt.tight_layout()
plt.show()

목표변수 vs 설명변수 관계 확인
- 범주형 설명변수 박스 플롯(Box Plot)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 범주형 변수 추출
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# Scatter Plot
n_cols = 2
n_rows = int(np.ceil(len(cat_cols) / n_cols))

plt.figure(figsize=(14, 5 * n_rows))

for i, col in enumerate(cat_cols, 1):
    plt.subplot(n_rows, n_cols, i)

    sns.boxplot(
        data=df,
        x=col,
        y="price"
    )

    plt.title(f"price by {col}")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### [Self-Practice]

- 실습 주제 : 파생변수를 신규로 추가해서 EDA 분석 재현하기
- 실습 조건
  - 파생 변수 1: 주택의 지하실 유무를 나타내는 범주형 변수 has_basement를 생성 (basement_area가 0보다 크면 1, 0이면 0)  
  - 파생 변수 2: 토지 면적 대비 건물 밀도를 나타내는 연속형 변수 living_lot_ratio 를 생성 (living_area / lot_area)
- 실습 시나리오
   - 시나리오 1) 위 조건으로 파생변수 2개를 만드는 코드 생성
   - 시나리오 2) 파생변수 2개가 포함된 df 데이타셋을 동일한 방식으로 EDA 수행
   - 시나리오 3) EDA 분석 결과 해석하기 (추가된 파생변수 중심으로)

# 파생변수 추가 (Self-Practice)
시나리오 1) 파생변수 2개 생성
- has_basement : 지하실 유무 (범주형, basement_area > 0 → 1, else 0)
- living_lot_ratio : 건물 밀도 (연속형, living_area / lot_area)

In [ ]:
# 파생변수 1: 지하실 유무 (범주형)
df["has_basement"] = (df["basement_area"] > 0).astype(int).astype("category")

# 파생변수 2: 건물 밀도 (연속형) = 주거 면적 / 토지 면적
df["living_lot_ratio"] = df["living_area"] / df["lot_area"]

df[["basement_area", "has_basement", "living_area", "lot_area", "living_lot_ratio"]].head()

컬럼정보 재확인 (has_basement: category, living_lot_ratio: float64)

In [ ]:
df.info()

시나리오 2) EDA 재현 (파생변수 포함 df 기준, 위와 동일한 방식)

연속형 변수 기본 분포값 확인 (living_lot_ratio 포함)

In [ ]:
import numpy as np

# 연속형 변수 분포 확인 (living_lot_ratio 자동 포함)
df.select_dtypes(include=[np.number]).describe()

연속형 변수 분포 그래프 확인
 - 히스토그램 (Histogram) 차트

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 연속형 변수 자동 추출
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("연속형 변수:", num_cols)

# 빈도 분포 히스토그램 (Frequency Histogram)
n = len(num_cols)
cols = 3
rows = (n // cols) + (n % cols > 0)

plt.figure(figsize=(cols*5, rows*4))

for i, col in enumerate(num_cols, 1):
    plt.subplot(rows, cols, i)
    plt.hist(df[col].dropna(), bins=20)
    plt.title(f"{col} Frequency")
    plt.xlabel(col)
    plt.ylabel("Count")

plt.tight_layout()
plt.show()

 - 박스 플롯(Box Plot) : 사분위수 범위 (IQR), 이상치 후보 확인

In [ ]:
import numpy as np

# 연속형 변수 자동 추출
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("연속형 변수:", num_cols)

# 박스 플롯(Box Plot)
cols = 3
rows = (len(num_cols) // cols) + (len(num_cols) % cols > 0)

plt.figure(figsize=(cols*5, rows*4))

for i, col in enumerate(num_cols, 1):
    plt.subplot(rows, cols, i)
    plt.boxplot(df[col].dropna())
    plt.title(col)
    plt.ylabel("Value")

plt.tight_layout()
plt.show()

 - 왜도, 첨도

In [ ]:
import numpy as np
import pandas as pd

# 연속형 변수 자동 추출
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("연속형 변수:", num_cols)

# 왜도 & 첨도 계산
skew_kurt = pd.DataFrame({
    "Skewness": df[num_cols].skew(),
    "Kurtosis": df[num_cols].kurt()
})

print(skew_kurt.sort_values(by="Skewness", ascending=False))

범주형 변수 기본 분포값 확인 (has_basement 포함)

In [ ]:
import pandas as pd

# 범주형 변수 자동 추출
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 범주 건수 & 비율 계산
for col in cat_cols:
    print(f"\n[{col}]")
    dist = df[col].value_counts(dropna=False)
    ratio = df[col].value_counts(normalize=True, dropna=False) * 100
    result = pd.DataFrame({
        "count": dist,
        "ratio_%": ratio.round(2)
    })
    print(result)

범주형 변수 분포 그래프 확인
 - 히스토그램 (Histogram) 차트

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 범주형 변수 찾기
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 시각화
for col in cat_cols:
    plt.figure(figsize=(10, 6))

    df[col].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')

    plt.title(f'{col} 빈도 분포', fontsize=14, fontweight='bold')
    plt.xlabel(col, fontsize=12)
    plt.ylabel('빈도', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

목표변수 vs 설명변수 관계 확인
- 연속형 설명변수 산점도(Scatter Plot) - living_lot_ratio 포함

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 연속형 변수 추출
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# 목표변수(price) 제외
feature_cols = [col for col in num_cols if col != "price"]

# Scatter Plot
n_cols = 3
n_rows = int(np.ceil(len(feature_cols) / n_cols))
plt.figure(figsize=(15, 4 * n_rows))

for i, col in enumerate(feature_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.scatterplot(data=df, x=col, y="price", alpha=0.5)
    plt.title(f"price vs {col}")

plt.tight_layout()
plt.show()

목표변수 vs 설명변수 관계 확인
- 범주형 설명변수 박스 플롯(Box Plot) - has_basement 포함

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 범주형 변수 추출
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

n_cols = 2
n_rows = int(np.ceil(len(cat_cols) / n_cols))

plt.figure(figsize=(14, 5 * n_rows))

for i, col in enumerate(cat_cols, 1):
    plt.subplot(n_rows, n_cols, i)

    sns.boxplot(
        data=df,
        x=col,
        y="price"
    )

    plt.title(f"price by {col}")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

시나리오 3) EDA 분석 결과 해석 (파생변수 중심)

**1. has_basement (지하실 유무)**
- 분포 : 지하실 없음 600건(60%), 있음 400건(40%) → 두 그룹 모두 표본 수가 충분해 비교 분석에 안정적
- price by has_basement (Box Plot) : 지하실 없음 평균 488,535 / 중앙값 386,350, 지하실 있음 평균 608,150 / 중앙값 496,750
  → 지하실이 있는 주택이 평균 약 24%, 중앙값 기준 약 29% 더 비쌈. `basement_area`를 연속형 그대로 쓰면 0이 60%를 차지해 분포가 왜곡되지만, 유무만 이진화하니 가격 차이가 뚜렷하게 드러남 → 파생변수로서 유의미

**2. living_lot_ratio (건물 밀도 = living_area / lot_area)**
- 분포(describe) : 평균 0.284, 중앙값 0.231, 최솟값 -0.021, 최댓값 1.743 → **최솟값이 음수**로 나타남. 이는 `living_area`에 음수 값(이상 데이터)이 존재하기 때문 (뒤쪽 "비정상 데이터 처리" 단계에서 `living_area < 0 → Missing` 처리로 해결됨). 즉 이 파생변수를 통해 원본 데이터의 이상치를 EDA 단계에서 미리 포착한 셈
- 히스토그램/왜도·첨도 : Skewness ≈ 2.23, Kurtosis ≈ 7.69 → 강한 오른쪽 꼬리(우측 치우침)와 뾰족한 분포. 대부분 주택은 0.1~0.4 사이(건물이 대지의 10~40%)에 몰려 있고, 일부 좁은 대지에 큰 건물을 올린 주택(비율 > 1)이 소수 존재
- price와의 상관관계 : corr(living_lot_ratio, price) ≈ 0.12 (약한 양의 상관) → 건물 밀도가 높을수록 가격이 소폭 상승하는 경향은 있으나, `living_area`나 `grade` 단독보다는 설명력이 약함. 반면 corr(living_lot_ratio, lot_area) ≈ -0.33으로, 대지가 넓을수록 오히려 건물 밀도는 낮아지는 경향(교외의 넓은 대지 주택은 건물이 대지에 비해 작음)이 확인됨

**종합** : `has_basement`는 가격과의 관계가 뚜렷해 회귀모델의 설명변수로 유용할 것으로 기대됨. `living_lot_ratio`는 가격 설명력은 약하지만, 입지·주거 형태(도심 밀집형 vs 교외 저밀도형)를 구분하는 보조 지표이자 데이터 품질 이슈(음수 living_area)를 조기에 발견하는 데 기여함.

# 데이터 전처리
비정상 데이터 처리

In [ ]:
import numpy as np

# living_area < 0  → Missing
df.loc[df["living_area"] < 0, "living_area"] = np.nan

# Data Partition
 - 목적 : 모델이 학습에 사용하지 않은 새로운 데이터에서도 잘 동작하는지 검증하기 위함
 - Train  : Test 비율 (7 : 3)
- Train Set : 모델 학습용, Test Set : 모델 평가용
- 데이타 변환시 변환 통계량은 Train 데이터에서 계산하고, Test 데이터에는 그 기준을 적용해야 함 (Data Leakage 문제)

데이타 분할

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.3,     # test 30%
    random_state=42,   # 재현성
    shuffle=True       # 섞어서 분리
)

원본 데이타 복제

In [ ]:
df_train_raw = df_train.copy()
df_test_raw  = df_test.copy()

# Missing 처리
Missing 데이터 확인

In [ ]:
df_train.isnull().sum()

Missing 데이터 변환
- 연속형 변수 대체 : median

In [ ]:
import numpy as np

# 연속형 변수만 선택
num_cols = df_train.select_dtypes(include=[np.number]).columns

# train 기준 중앙값 계산
median_values = df_train[num_cols].median()

# train, test 모두 중앙값으로 결측치 대체
df_train[num_cols] = df_train[num_cols].fillna(median_values)
df_test[num_cols]  = df_test[num_cols].fillna(median_values)

 - 범주형 변수 대체 : New class ('Unknown')

In [ ]:
# 범주형 변수만 선택
cat_cols = df_train.select_dtypes(include=['object', 'category']).columns

# 신규 범주 (Unknown) 로 대체
for col in cat_cols:
    # train
    if df_train[col].dtype.name == 'category':
        df_train[col] = df_train[col].cat.add_categories('Unknown')
    df_train[col] = df_train[col].fillna('Unknown')

    # test
    if df_test[col].dtype.name == 'category':
        df_test[col] = df_test[col].cat.add_categories('Unknown')
    df_test[col] = df_test[col].fillna('Unknown')

Missing 데이터 재확인

In [ ]:
df_train.isnull().sum()

# Outlier 처리
 - 이상치 탐지 방법 : Z-score 기반 탐지 (|Z| > 3)
 - 이상치 대체 방법 : Z-score 경계값으로 대체
 - 이상치 탐지 대상 변수 : 연속형 설명변수 *종속변수가 이상치로 의심되는 경우 처리 -> 모델 안정성을 위해 분석 제외

In [ ]:
import numpy as np

# 연속형 변수 추출
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()

# 목표변수 제외
num_cols.remove("price")

# Z-score 기반 이상치 처리
for col in num_cols:

    # Train 기준 평균, 표준편차
    mean = df_train[col].mean()
    std = df_train[col].std()

    # |Z| > 3 경계값
    lower = mean - 3 * std
    upper = mean + 3 * std

    # Train 데이터 이상치 건수
    train_outlier_cnt = ((df_train[col] < lower) | (df_train[col] > upper)).sum()

    # Test 데이터 이상치 건수
    test_outlier_cnt = ((df_test[col] < lower) | (df_test[col] > upper)).sum()

    print(
        f"{col:20s} "
        f"Train:{train_outlier_cnt:3d}건 "
        f"Test:{test_outlier_cnt:3d}건"
    )

    # Train 데이터 Capping
    df_train[col] = np.clip(df_train[col], lower, upper)

    # Test 데이터도 동일 기준 적용
    df_test[col] = np.clip(df_test[col], lower, upper)

### [Self-Practice]

- 실습 주제 : 이상치 처리 방법 비교해 보기
- 실습 시나리오
   - 시나리오) Z-score 방법 대신 IQR 방법 적용 후 Train 데이타와 Test 데이타 이상치 건수 상호 비교하기

# 이상치 처리 방법 비교 (Self-Practice)
시나리오) Z-score 방법 대신 IQR 방법 적용 후 Train / Test 이상치 건수 상호 비교

- IQR 방법 : Q1 - 1.5×IQR ~ Q3 + 1.5×IQR 범위를 벗어나면 이상치 (IQR = Q3 - Q1)
- 주의 : 위 Z-score 셀이 이미 df_train / df_test 에 Capping 을 적용했으므로, 공정한 비교를 위해 **Capping 이전 데이터(df_train_raw 복제 + 동일한 median 대체)** 에서 두 방법을 나란히 계산함
- 경계값은 두 방법 모두 Train 기준으로 계산하고 Test 에 동일 적용 (Data Leakage 방지)

In [ ]:
import numpy as np

# =========================
# Capping 이전 데이터 재구성 (원본 복제 + median 대체)
# =========================
df_train_cmp = df_train_raw.copy()
df_test_cmp  = df_test_raw.copy()

num_cols = df_train_cmp.select_dtypes(include=[np.number]).columns
median_values = df_train_cmp[num_cols].median()
df_train_cmp[num_cols] = df_train_cmp[num_cols].fillna(median_values)
df_test_cmp[num_cols]  = df_test_cmp[num_cols].fillna(median_values)

# 연속형 변수 추출, 목표변수 제외
num_cols = df_train_cmp.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove("price")

# =========================
# Z-score vs IQR 이상치 건수 비교
# =========================
print(f"{'변수':18s} | {'Z-score Train':>13s} {'Test':>5s} | {'IQR Train':>9s} {'Test':>5s}")
print("-" * 65)

for col in num_cols:

    # --- Z-score 기준 (|Z| > 3) ---
    mean = df_train_cmp[col].mean()
    std  = df_train_cmp[col].std()
    z_lower, z_upper = mean - 3 * std, mean + 3 * std

    z_train_cnt = ((df_train_cmp[col] < z_lower) | (df_train_cmp[col] > z_upper)).sum()
    z_test_cnt  = ((df_test_cmp[col]  < z_lower) | (df_test_cmp[col]  > z_upper)).sum()

    # --- IQR 기준 (1.5 × IQR) ---
    q1 = df_train_cmp[col].quantile(0.25)
    q3 = df_train_cmp[col].quantile(0.75)
    iqr = q3 - q1
    i_lower, i_upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    i_train_cnt = ((df_train_cmp[col] < i_lower) | (df_train_cmp[col] > i_upper)).sum()
    i_test_cnt  = ((df_test_cmp[col]  < i_lower) | (df_test_cmp[col]  > i_upper)).sum()

    print(f"{col:18s} | {z_train_cnt:>13d} {z_test_cnt:>5d} | {i_train_cnt:>9d} {i_test_cnt:>5d}")

print("-" * 65)
print(f"Train {len(df_train_cmp)}건 / Test {len(df_test_cmp)}건")

비교 결과 해석

| 변수 | Z-score (Train/Test) | IQR (Train/Test) | 비고 |
|---|---|---|---|
| bedrooms | 3 / 2 | 15 / 7 | IQR이 5배 더 탐지 |
| living_area | 12 / 3 | 28 / 12 | IQR이 2배 이상 |
| lot_area | 12 / 10 | **80 / 37** | 차이 최대 (왜도 6.1) |
| floors | 12 / 7 | **0 / 0** | IQR은 전혀 탐지 못함 |
| above_area | 9 / 5 | 21 / 9 | |
| basement_area | 9 / 6 | 19 / 8 | |
| yr_built | 0 / 0 | 0 / 0 | 대칭 분포 → 둘 다 없음 |
| house_age | 0 / 0 | 0 / 0 | 〃 |
| room_density | 10 / 5 | 39 / 18 | |
| living_lot_ratio | 15 / 10 | 37 / 18 | |

**1. 전반적으로 IQR이 Z-score보다 훨씬 많은 이상치를 탐지함**
- Z-score의 평균±3σ 기준은 극단값 자체가 평균과 표준편차를 부풀리기 때문에 경계가 넓어져서 탐지가 느슨해짐 (특히 왜도가 큰 lot_area: 12건 vs 80건)
- IQR은 사분위수 기반이라 극단값의 영향을 받지 않아(robust) 치우친 분포에서도 일관된 기준을 유지함

**2. 예외 — floors는 반대로 IQR이 0건**
- floors는 1, 1.5, 2 같은 소수의 이산값이라 Q1=1, Q3=2 로 IQR 범위가 모든 값을 포함해버림
- 반면 Z-score는 3층(빈도 낮음)을 이상치로 탐지 → **이산형/범주 성격 변수에는 두 방법 모두 부적합**하며, 이런 변수는 이상치 처리 대상에서 제외하는 것이 타당

**3. Train vs Test 건수 비교**
- 모든 변수에서 Test 건수가 Train의 약 40~50% 수준 (표본 비율 7:3과 대체로 일치) → Train 기준 경계값이 Test에도 무리 없이 적용됨을 확인
- 특정 변수에서 Test 비율이 비정상적으로 높다면 Train/Test 분포가 다르다는 신호 (여기서는 해당 없음)

**결론** : 이 데이터처럼 왜도가 큰 변수가 많으면 IQR이 이상치를 더 민감하게 탐지함. 다만 어느 쪽이 "정답"이 아니라, 탐지 건수가 크게 달라지므로 **처리 방법 선택이 이후 분포 변환·모델 성능에 영향을 주는 의사결정**임을 확인하는 것이 이 실습의 핵심.

# 분포 변환
 - 분포 변환 방법 : Log 변환
 - 변환 대상 : 연속형 설명변수

In [ ]:
# 시나리오 1) Log 변환 직전의 Train 왜도값 확인
# (아래 Log 변환 셀이 df_train 을 덮어쓰기 때문에, 변환 전 상태를 미리 저장해 둠)
num_cols_before = df_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols_before.remove("price")

skew_before = df_train[num_cols_before].skew()
print(skew_before.sort_values(ascending=False))

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 변수 분리
# ==========================================

# 전체 연속형 변수 추출 후 'price' 제외
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
if "price" in num_cols:
    num_cols.remove("price")

# 범주형 변수 추출
cat_cols = df_train.select_dtypes(include=["object", "category"]).columns.tolist()


# ==========================================
# Log Transformation 함수 정의
# ==========================================

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


# ==========================================
# Train 및 Test 데이터 변환 적용
# ==========================================

# Train 연속형 변수 변환
train_log = df_train[num_cols].apply(signed_log1p)

# Test 연속형 변수 변환
test_log = df_test[num_cols].apply(signed_log1p)


# ==========================================
# 데이터 병합 (변환된 변수 + 원본 price + 범주형)
# ==========================================

# df_train
df_train = pd.concat(
    [
        train_log,              # Log 변환된 연속형 피처들
        df_train[["price"]],    # 원본 타겟 변수 (price)
        df_train[cat_cols]      # 범주형 변수들
    ],
    axis=1
)

# df_test
df_test = pd.concat(
    [
        test_log,               # Log 변환된 연속형 피처들
        df_test[["price"]],     # 원본 타겟 변수 (price)
        df_test[cat_cols]       # 범주형 변수들
    ],
    axis=1
)

### [Self-Practice]

- 실습 주제 : 분포 변환 효과 확인하기
- 실습 시나리오
   - 시나리오 1) Log 변환 직전의 Train 데이타에서 왜도값 확인하기
   - 시나리오 2) Log 변환 직후의 Train 데이타에서 왜도값 확인 후 변환전 왜도값과 비교하기

In [ ]:
# 시나리오 2) Log 변환 직후의 Train 왜도값 확인 후 변환 전과 비교
# (이 시점의 df_train 은 이미 위 Log 변환 셀에 의해 log 스케일로 바뀐 상태)
skew_after = df_train[num_cols_before].skew()

skew_cmp = pd.DataFrame({
    "Skew_Before": skew_before,
    "Skew_After": skew_after
})
skew_cmp["Abs_Before"] = skew_cmp["Skew_Before"].abs()
skew_cmp["Abs_After"] = skew_cmp["Skew_After"].abs()
skew_cmp["Improved"] = skew_cmp["Abs_After"] < skew_cmp["Abs_Before"]

print(skew_cmp.sort_values("Abs_Before", ascending=False))

비교 결과 해석

| 변수 | Skew 변환前 | Skew 변환後 | 개선 |
|---|---|---|---|
| lot_area | 3.47 | 0.53 | ✅ 크게 개선 |
| living_lot_ratio | 1.45 | 1.02 | ✅ 개선 |
| basement_area | 1.37 | 0.47 | ✅ 크게 개선 |
| above_area | 1.11 | 0.20 | ✅ 크게 개선 |
| living_area | 1.07 | -0.01 | ✅ 크게 개선 (거의 대칭) |
| room_density | 0.97 | **-10.84** | ❌ 크게 악화 |
| floors | 0.67 | 0.48 | ✅ 개선 |
| bedrooms | 0.59 | -0.67 | △ 부호만 반전, 절대값은 비슷 |
| yr_built | -0.54 | -0.57 | △ 거의 변화 없음 |
| house_age | 0.54 | -1.33 | ❌ 악화 |

**1. 대부분의 변수는 Log 변환으로 왜도가 뚜렷이 개선됨**
- `lot_area`(3.47→0.53), `basement_area`(1.37→0.47), `above_area`(1.11→0.20), `living_area`(1.07→-0.01)처럼 원래 우측으로 크게 치우쳐 있던 면적 계열 변수들이 log 변환 후 0에 가까워짐. 값의 범위가 넓고 항상 양수인 변수에 log 변환이 잘 작동하는 전형적인 사례

**2. `room_density`는 오히려 왜도가 폭발적으로 악화됨 (0.97 → -10.84)**
- 원인 : `room_density = living_area / bedrooms`는 파생변수 생성 시점에 계산되는데, 그보다 뒤에 실행되는 "living_area < 0 → Missing" 정제가 `room_density`에는 소급 적용되지 않음. 그 결과 `room_density`에 living_area 음수값에서 비롯된 **음수 이상치(최소값 -33.3)**가 그대로 남아있음
- 대부분 값은 400~1200 사이에 촘촘히 몰려 log 변환 후 6.2~6.5 근처로 더 좁게 압축되는데, 소수의 음수값은 `signed_log1p`가 `sign(x)×log1p(|x|)`라서 부호가 반전된 채(-3.5 부근) 남아 정상값 군집과 큰 거리가 생김. 표준편차가 작아진 상태에서 극단값 몇 개가 남으니 왜도 공식(3차 모멘트/표준편차³)이 오히려 폭증함
- 시사점 : **파생변수는 재료가 되는 원본 컬럼을 정제하기 전에 만들면, 원본만 고쳐도 파생변수에는 오염이 남는다**는 것을 보여주는 사례. `room_density`도 `living_area` 정제 이후 재계산하거나, 정제 순서를 앞당겨야 함

**3. `bedrooms`, `house_age`, `yr_built`는 개선되지 않거나 부호만 바뀜**
- 이 세 변수는 애초에 왜도가 크지 않았던(0.5~0.6 수준) 변수라 Log 변환의 효과가 제한적이거나 오히려 미세하게 역효과가 남
- Log 변환은 "우측으로 크게 치우친 분포"에 효과적인 도구이지, 모든 변수에 기계적으로 적용해서 좋은 게 아님을 보여줌

**결론** : Log 변환은 면적처럼 왜곡이 크고 항상 양수인 변수에는 효과적이지만, 만능이 아니다. 특히 이번 케이스처럼 **파생변수가 정제 이전의 원본값을 참조하고 있으면 이상치가 은닉된 채로 남아 변환 후 오히려 악화**될 수 있으므로, 변환 전후 왜도를 반드시 비교 검증해야 한다는 것이 이 실습의 핵심 교훈.

# 데이터 Scaling
 - Scaling 방법 : Min-Max 변환
- Scaling 대상 변수 : 연속형 설명변수

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

# =========================
# 변수 분리
# =========================

# 연속형 변수
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()

# price 제외
num_cols.remove("price")

# 범주형 변수
cat_cols = df_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# =========================
# Min-Max Scaling
# =========================

scaler = MinMaxScaler()

# Train 기준 학습
train_scaled = pd.DataFrame(
    scaler.fit_transform(df_train[num_cols]),
    columns=num_cols,
    index=df_train.index
)

# Test는 동일 기준 적용
test_scaled = pd.DataFrame(
    scaler.transform(df_test[num_cols]),
    columns=num_cols,
    index=df_test.index
)

# =========================
# 데이터 병합
# =========================

df_train = pd.concat(
    [
        train_scaled,            # 표준화된 연속형 변수
        df_train[["price"]],     # 원본 price
        df_train[cat_cols]       # 범주형 변수
    ],
    axis=1
)

df_test = pd.concat(
    [
        test_scaled,
        df_test[["price"]],
        df_test[cat_cols]
    ],
    axis=1
)

# 결과 확인
print(df_train.head())

### [Self-Practice]

- 실습 주제 : Scaling 방법 비교해 보기
- 실습 시나리오
   - 시나리오 1) Min-Max 변환 후 Train 데이타 기준으로 평균/ 표준편차/ 최소값/ 최대값 확인하기
   - 시나리오 2) Min-Max 대신 Z-Score 변환 후 Train 데이타 기준으로 평균/ 표준편차/ 최소값/ 최대값 확인하고, Min-Max 변환 결과와 비교하기

# 상관분석
- 다중공선성 방지, 차원의 저주 문제 방지, 설명변수 축소 목적   
\* [참고] 차원의 저주: 변수의 개수가 너무 많은 경우 예측 성능이 떨어지는 현상    


상관계수 산출

In [ ]:
# 연속형 변수 추출
num_cols = df_train.select_dtypes(include=[np.number])

# 상관계수
corr_matrix = num_cols.corr()
print(corr_matrix)

히트맵 시각화

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix (Numeric Variables)")
plt.show()

# 회귀 분석 준비

X, y 분리

In [ ]:
# train
Y_train = df_train['price']
X_train = df_train.drop(columns=['price'])

# test
Y_test = df_test['price']
X_test = df_test.drop(columns=['price'])

One-Hot Encoding
- k-1 개 dummy 변수 생성

In [ ]:
# 범주형 컬럼 선택
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

# dummy 변수화 + 연속형 자동 병합
# Train Set
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)   # True: k-1개 dummy, False: One-Hot Encoding

# Test Set
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# test를 train 컬럼 구조에 맞추기 (없는 컬럼은 0으로, train에 없는 test컬럼은 제거)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

 - X_train 변수 타입 확인

In [ ]:
X_train.dtypes

- Bool Type -> Int Type 변환  *statsmodels 라이브러리 적용 위함

In [ ]:
# bool 컬럼 찾기
bool_cols = X_train.select_dtypes(include=['bool']).columns

# train, test 모두 변환
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

# 회귀 분석
 - statsmodels 라이브러리 사용, API 문서 : https://www.statsmodels.org/stable/index.html
 - statsmodels 라이브러리 : p-value, F 통계량 등 출력됨, 주로 통계적 추론 목적  
  \* [참고] sklearn.linear_model 라이브러리: p-value, F 통계량 등 출력안됨, 예측 목적  

학습 모델 생성
- statsmodels

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error

# ==========================================
# Train 데이터로 회귀모델 생성
# ==========================================

# 절편(constant) 추가
X_train_const = sm.add_constant(X_train, has_constant="add")

# 회귀모델 생성
ols_model = sm.OLS(Y_train, X_train_const).fit()

# 회귀분석 결과
print(ols_model.summary())

In [ ]:
# ==========================================
# Train 데이터 예측
# ==========================================

# 예측값 생성
train_pred = ols_model.predict(X_train_const)

# MAPE 계산
train_mape = mean_absolute_percentage_error(Y_train, train_pred) * 100

print(f"Train MAPE : {train_mape:.2f}%")

In [ ]:
# ==========================================
# Test 데이터 예측
# ==========================================

# 절편 추가
X_test_const = sm.add_constant(X_test, has_constant="add")

# 예측값 생성
test_pred = ols_model.predict(X_test_const)

# MAPE 계산
test_mape = mean_absolute_percentage_error(Y_test, test_pred) * 100

print(f"Test MAPE : {test_mape:.2f}%")

실제값 vs 예측값 비교
- 원래 Test 데이터에 예측값 병합하기

In [ ]:
df_test_raw["y_pred"] = test_pred
df_test_raw

- 실제값 vs 예측값 시각화 비교

In [ ]:
import matplotlib.pyplot as plt

# 실제값과 예측값
actual = df_test_raw["price"]
pred = df_test_raw["y_pred"]

# 그래프 크기
plt.figure(figsize=(7, 7))

# 산점도
plt.scatter(actual, pred, alpha=0.6)

# y=x 기준선
min_val = min(actual.min(), pred.min())
max_val = max(actual.max(), pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--",
    linewidth=2,
    label="Ideal Prediction"
)

# 그래프 설정
plt.title("Actual vs Predicted Price")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

### [Self-Practice]

- 실습 주제 : 다중공선성 문제 해결과 변수 최적화 실험
- 실습 시나리오
   - 시나리오 1) 상관분석 결과에서 상관계수(|r|) ≥ 0.8 인 변수 중 하나를 제거하여 회귀모델을 생성한 후, 제거 전후의 Train, Test 데이터 MAPE 값을 비교함
  - 시나리오 2) 회귀분석 결과에서 변수 유의성 수준(P>|t|) ≥ 0.05 인 셜명변수를 제거하여 회귀모델을 생성한 후, 제거 전후의 후 Train 데이터 R-squared, Adj. R-squared, MAPE 값을 비교함
  - 시나리오 3) 파생변수 생성, 데이터 전처리(결측치 대체, 이상치 처리, 스케일링, 분포 변환), 상관분석 및 변수 유의성 검증을 통해 회귀모델을 반복적으로 개선하고, Test 데이터 기준 MAPE를 최소화하는 모델 파이프라인 생성하기

# Baseline 모델 평가
 - Naive Model  *가장 단순하고 직관적인 방법으로 만든 기준 모델

In [ ]:
import numpy as np

# train 타겟 평균
y_mean = Y_train.mean()
print(f"Y Mean : {y_mean:.2f}")

# test 개수만큼 평균값 복제
y_pred_baseline = np.full(shape=len(Y_test), fill_value=y_mean)

# mape 계산
mape = mean_absolute_percentage_error(Y_test, y_pred_baseline) * 100
print(f"Baseline MAPE : {mape:.2f}%")